In [1]:
import os
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [33]:

@tool
def check_inventory(product: str) -> str:
    """
    Check the current inventory quantity for a product.

    Args:
        product: The name of the product whose inventory availability
            needs to be checked.

    Returns:
        A string indicating the number of units currently available
        for the requested product.

    Raises:
        ValueError: If the requested product does not exist in the
            inventory.
    """

    inventory = {
        "iphone 17": 15,
        "samsung s26": 8,
        "macbook air": 12,
        "ps5": 4,
    }

    product_key = product.lower()
    quantity = inventory.get(product_key)

    if quantity is None:
        return "Product '{product}' was not found in inventory."

    return f"{product} has {quantity} units currently in stock."


@tool
def get_weather(city: str) -> str:
    """
    Get the current weather information for a city.

    Args:
        city: The name of the city for which weather information
            is requested. Use the city name without the country
            unless clarification is required.

    Returns:
        A string containing the current weather conditions and
        temperature for the requested city.

    Raises:
        ValueError: If weather information is not available for
            the requested city.
    """

    weather_data = {
        "Chennai": "31°C, partly cloudy",
        "Bangalore": "21°C, cloudy",
        "Mumbai": "11°C, humid",
        "Delhi": "1°C, snowy",
    }

    weather = weather_data.get(city)

    if weather is None:
        return "Weather information is not available for {city}."

    return f"Current weather in {city}: {weather}"


@tool
def convert_currency(
    amount: float,
    from_currency: str,
    to_currency: str
) -> str:
    """
    Convert a monetary amount from one currency to another.

    Args:
        amount: The amount of money to convert. Must be greater than
            or equal to zero.
        from_currency: The three-letter ISO currency code of the
            currency being converted, such as "USD", "EUR", or "GBP".
        to_currency: The three-letter ISO currency code of the
            target currency, such as "INR", "USD", or "EUR".

    Returns:
        A formatted string containing the original amount, exchange
        rate, and converted amount.

    Raises:
        ValueError: If the amount is negative or the requested
            currency conversion is not supported.
    """

    if amount < 0:
        raise ValueError("Amount cannot be negative.")

    rates = {
        ("INR", "USD"): 50,
        ("INR", "EUR"): 100,
        ("INR", "GBP"): 150,
    }

    from_currency = from_currency.upper()
    to_currency = to_currency.upper()

    rate = rates.get((from_currency, to_currency))

    if rate is None:
        return "Conversion from {from_currency} to {to_currency} is not supported."

    converted_amount = amount * rate

    return (
        f"{amount:.2f} {from_currency} = "
        f"{converted_amount:.2f} {to_currency} "
        f"(exchange rate: {rate})"
    )

@tool
def get_employee(employee_name: str) -> str:
    """
    Retrieve information about an employee using their name.

    Args:
        employee_name: The full or commonly used name of the employee
            whose information is being requested.

    Returns:
        A string containing the employee's department and job role.

    Raises:
        ValueError: If an employee with the specified name cannot
            be found.
    """

    employees = {
        "Prashanth": {
            "department": "Data Science",
            "role": "Senior Data Scientist",
        },
        "Priya": {
            "department": "Finance",
            "role": "Financial Analyst",
        },
        "Divya": {
            "department": "Engineering",
            "role": "Software Engineer",
        },
    }

    employee = employees.get(employee_name)

    if employee is None:
        return "No employee found with the name '{employee_name}'."

    return (
        f"{employee_name} works in the "
        f"{employee['department']} department as a "
        f"{employee['role']}."
    )

In [34]:
agent = create_agent(
    model = "gpt-5.4-mini",
    tools = [get_employee,convert_currency,get_weather,check_inventory],
    checkpointer=InMemorySaver(),
    middleware= [SummarizationMiddleware(
        model="gpt-5.4-mini",
        trigger = ("messages",10),
        keep = ("messages",6)
    )],
    system_prompt= "You are a helpful assistant"
)

In [35]:
config_1 = {"configurable": {"thread_id": "001"}}

In [37]:
while True:
    user_input = input("User: ")
    print(f"User: {user_input}")
    if user_input.lower() in ["exit", "quit"]:
        print("Exiting the assistant. Goodbye!")
        break

    response_1 = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config_1
    )
    print(f"Assistant: {response_1["messages"][-1].content}")

User: convert 2 INR to USD
Assistant: 2.00 INR = 100.00 USD (exchange rate: 50)
User: 5234 INR to GBP?
Assistant: 5234.00 INR = 785100.00 GBP (exchange rate: 150)
User: get weather in Chennai, Mumbai
Assistant: Current weather in Chennai: 31°C, partly cloudy  
Current weather in Mumbai: 11°C, humid
User: get weather from chennai london and banglore
Assistant: Here are the weather results:

- Chennai: 31°C, partly cloudy
- London: weather information is not available
- Banglore: weather information is not available
User: how about Bangalore and Delhi
Assistant: Here you go:

- Bangalore: 21°C, cloudy
- Delhi: 1°C, snowy
User: if i sell all my PS5 in my inventory at 40 INR how many GBP will i get
Assistant: You have 4 PS5s in inventory.

At 40 INR each, that’s 160 INR total.

Converted to GBP: **24000.00 GBP**.
User: quit
Exiting the assistant. Goodbye!
